# Reproduce and ablate a denoising diffusion model

Reproduce the epsilon-prediction objective, the reverse sampler and one of the paper's ablations from
Ho, Jain and Abbeel, [Denoising Diffusion Probabilistic Models](https://arxiv.org/abs/2006.11239)
(Algorithms 1 and 2, Section 3.2). The paper fixes the reverse variance to $\sigma_t^2 I$ and reports
that $\sigma_t^2=\beta_t$ and $\sigma_t^2=\tilde\beta_t=\frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t}\beta_t$
gave similar results. You test that claim on matched weights.

The notebook has two parts.

- **Part A (CPU core, about 1 minute).** A two-dimensional synthetic mixture, a one-hidden-layer NumPy
  denoiser and a short linear schedule. Every formula is checked against analytic fixtures here.
  Two matched ablations: time-conditioned versus time-blind training (same initialization, data and
  optimizer steps), and the reverse variance $\beta_t$ versus $\tilde\beta_t$ on the same trained weights
  with the same noise draws.
- **Part B (Colab GPU milestone, about 45 minutes on a free T4).** An epsilon-prediction DDPM on MNIST
  with a small U-Net, $T=1000$ and the paper's linear $\beta$ schedule, sample grids, the same variance
  ablation on the trained weights, and a Fréchet distance in the feature space of a small MNIST
  classifier trained in the notebook.

Neither part reproduces the paper's CIFAR-10 FID or Inception scores. Replace each TODO and run the
assertions. Record failures and write the paper at the end.

In [ ]:
import platform
from dataclasses import asdict, dataclass
from time import perf_counter

import numpy as np

@dataclass(frozen=True)
class Protocol:
    steps: int = 40
    beta_first: float = 0.02
    beta_last: float = 0.18
    width: int = 48
    updates: int = 850
    batch_size: int = 96
    learning_rate: float = 0.002
    eval_size: int = 400
    seeds: tuple = (11, 29, 47, 61, 83)
    evaluator_seed: int = 901

protocol = Protocol()
centers = np.array([[-2.,-1.],[2.,1.]])
print("Python", platform.python_version(), "NumPy", np.__version__)
print("Protocol:", asdict(protocol))

def draw_data(rng,size):
    labels = rng.integers(0,len(centers),size=size)
    return centers[labels] + 0.3*rng.standard_normal((size,2))


## 1. Implement the forward marginal

Implement forward_sample(clean, noise, alpha_bar) for aligned (batch, features) arrays.
alpha_bar is a length-batch vector of cumulative retentions in [0,1]. Broadcast each
retention down its feature row. Use sqrt(alpha_bar) times clean and sqrt(1-alpha_bar)
times noise. The training scaffold samples one timestep per example.


In [ ]:
def forward_sample(clean,noise,alpha_bar):
    raise NotImplementedError


In [ ]:
clean=np.array([[1.,2.],[3.,4.]])
noise=np.array([[5.,6.],[7.,8.]])
np.testing.assert_allclose(forward_sample(clean,noise,np.array([1.,0.])),[[1.,2.],[7.,8.]])
np.testing.assert_allclose(forward_sample(clean,noise,np.array([0.25,0.25])),
                           0.5*clean+np.sqrt(0.75)*noise)

print("OK")


## 2. Implement one DDPM reverse step and the two variance choices

Implement reverse_variance(beta, retained, previous_retained, choice). Return beta when choice is
"beta" and the posterior variance beta * (1-previous_retained)/(1-retained) when choice is
"posterior". Reject any other choice with ValueError.

Implement reverse_sample(state, predicted_noise, beta, retained, variance, noise). State and noise
arrays are aligned; scalar beta and cumulative retention name the current step. Compute the
epsilon-parameterized mean (state - beta * predicted_noise / sqrt(1-retained)) / sqrt(1-beta),
then add sqrt(variance) times the supplied noise. As in Algorithm 2, the sampler passes zero noise
at the final step, so the clean endpoint is the mean.

In [ ]:
def reverse_variance(beta,retained,previous_retained,choice):
    raise NotImplementedError

def reverse_sample(state,predicted_noise,beta,retained,variance,noise):
    raise NotImplementedError

In [ ]:
x=np.array([[2.,-1.]])
epsilon=np.array([[0.4,2.]])
x1=forward_sample(x,epsilon,np.array([0.75]))
np.testing.assert_allclose(reverse_variance(0.25,0.75,1.,"posterior"),0.,atol=1e-12)
np.testing.assert_allclose(reverse_variance(0.2,0.6,0.75,"posterior"),0.125)
np.testing.assert_allclose(reverse_variance(0.2,0.6,0.75,"beta"),0.2)
try:
    reverse_variance(0.2,0.6,0.75,"learned")
except ValueError:
    pass
else:
    raise AssertionError("reject an unknown variance choice")
np.testing.assert_allclose(reverse_sample(x1,epsilon,0.25,0.75,0.,epsilon),x)
mean=reverse_sample(x,epsilon,0.2,0.6,0.125,np.zeros_like(epsilon))
np.testing.assert_allclose(mean,(x-0.2*epsilon/np.sqrt(0.4))/np.sqrt(0.8))
for variance in [0.125,0.2]:
    draw=reverse_sample(x,epsilon,0.2,0.6,variance,np.ones_like(epsilon))
    np.testing.assert_allclose(draw-mean,np.sqrt(variance))

print("OK")

## 3. Fit a small denoiser with a matched time ablation

The shared NumPy scaffold uses tanh activations and Adam. The loss is mean squared error across
all sample coordinates. Time features are scaled timestep and its square. The blind condition
zeros both columns and leaves the model shape, optimizer and draw sequence unchanged. Training
data are sampled from the defined population with training-only seeds. The independent evaluation
set uses a separate RNG. The optimizer and narrow time encoding differ from the image system in
the paper. Read through the setup before training and record any changes.


In [ ]:
def schedule(config):
    betas=np.linspace(config.beta_first,config.beta_last,config.steps)
    retained=np.cumprod(1-betas)
    return betas,retained

def features(state,index,steps,conditioned):
    t=(index+1)/steps
    time=np.column_stack((t,t*t)) if conditioned else np.zeros((len(state),2))
    return np.column_stack((state,time))

def initial_model(rng,width):
    return [rng.normal(0,0.25,size=(4,width)),np.zeros(width),
            rng.normal(0,0.12,size=(width,2)),np.zeros(2)]

def predict(parameters,inputs):
    w1,b1,w2,b2=parameters
    hidden=np.tanh(inputs@w1+b1)
    return hidden@w2+b2,hidden

def train_denoiser(config,seed,conditioned):
    rng=np.random.default_rng(seed)
    params=initial_model(rng,config.width)
    first=[np.zeros_like(value) for value in params]
    second=[np.zeros_like(value) for value in params]
    _,retained=schedule(config)
    for update in range(1,config.updates+1):
        clean=draw_data(rng,config.batch_size)
        t=rng.integers(0,config.steps,size=config.batch_size)
        noise=rng.standard_normal(clean.shape)
        inputs=features(forward_sample(clean,noise,retained[t]),t,config.steps,conditioned)
        prediction,hidden=predict(params,inputs)
        score_gradient=2*(prediction-noise)/noise.size
        hidden_gradient=(score_gradient@params[2].T)*(1-hidden**2)
        grads=[inputs.T@hidden_gradient,hidden_gradient.sum(axis=0),
               hidden.T@score_gradient,score_gradient.sum(axis=0)]
        for j,grad in enumerate(grads):
            first[j]=0.9*first[j]+0.1*grad
            second[j]=0.999*second[j]+0.001*grad**2
            first_hat=first[j]/(1-0.9**update)
            second_hat=second[j]/(1-0.999**update)
            params[j]-=config.learning_rate*first_hat/(np.sqrt(second_hat)+1e-8)
    if not all(np.isfinite(value).all() for value in params):
        raise ValueError("training produced nonfinite weights")
    return params


## 4. Freeze evaluation and define success

The primary contrast is held-out epsilon MSE of the conditioned model minus that of the blind
model, paired by training seed. Lower is better. Before running the study, write the smallest
difference you consider meaningful and why. The sampling contrast is the paper's reverse-variance
ablation: squared MMD at one declared RBF bandwidth with $\sigma_t^2=\beta_t$ minus that with
$\sigma_t^2=\tilde\beta_t$. Both samplers use the same conditioned weights and the same noise draws
within each seed. It is a secondary, fixed-protocol comparison, conditional on the trained model.
The paper reports similar results for the two choices, so state in advance the largest difference
you would still call similar. A small number of seeds cannot establish robust
generalization. Do not tune the bandwidth or schedule after seeing the table.

The evaluator uses independent synthetic reference draws plus shared random numbers for the two
variance choices within each seed. Mode fraction uses nearest center and may count distant samples;
report it together with the kernel distance and inspect the two-dimensional samples.


In [ ]:
def generate(params,config,seed,choice):
    rng=np.random.default_rng(seed)
    betas,retained=schedule(config)
    state=rng.standard_normal((config.eval_size,2))
    for t in range(config.steps-1,-1,-1):
        inputs=features(state,np.full(len(state),t),config.steps,True)
        prediction,_=predict(params,inputs)
        previous=retained[t-1] if t else 1.
        variance=reverse_variance(betas[t],retained[t],previous,choice)
        noise=rng.standard_normal(state.shape) if t else np.zeros_like(state)
        state=reverse_sample(state,prediction,betas[t],retained[t],variance,noise)
    return state

def kernel_mean(x,y,bandwidth):
    squared=np.sum((x[:,None,:]-y[None,:,:])**2,axis=-1)
    return np.exp(-squared/(2*bandwidth**2)).mean()

def mmd_squared(x,y,bandwidth=1.):
    return kernel_mean(x,x,bandwidth)+kernel_mean(y,y,bandwidth)-2*kernel_mean(x,y,bandwidth)

def right_mode_fraction(samples):
    distances=np.sum((samples[:,None,:]-centers[None,:,:])**2,axis=-1)
    return np.mean(distances.argmin(axis=1)==1)

def run_experiment(config):
    rng=np.random.default_rng(config.evaluator_seed)
    clean=draw_data(rng,config.eval_size)
    reference=draw_data(rng,config.eval_size)
    step=rng.integers(0,config.steps,size=config.eval_size)
    noise=rng.standard_normal(clean.shape)
    _,retained=schedule(config)
    noisy=forward_sample(clean,noise,retained[step])
    rows=[]
    example_samples=None
    for seed in config.seeds:
        started=perf_counter()
        conditioned=train_denoiser(config,seed,True)
        blind=train_denoiser(config,seed,False)
        cond_prediction,_=predict(conditioned,features(noisy,step,config.steps,True))
        blind_prediction,_=predict(blind,features(noisy,step,config.steps,False))
        beta_samples=generate(conditioned,config,seed+1000,"beta")
        posterior_samples=generate(conditioned,config,seed+1000,"posterior")
        if example_samples is None:
            example_samples=(reference,beta_samples,posterior_samples)
        rows.append({
            "seed":seed,
            "conditioned_mse":float(np.mean((cond_prediction-noise)**2)),
            "blind_mse":float(np.mean((blind_prediction-noise)**2)),
            "beta_mmd2":float(mmd_squared(beta_samples,reference)),
            "posterior_mmd2":float(mmd_squared(posterior_samples,reference)),
            "beta_right_mode":float(right_mode_fraction(beta_samples)),
            "posterior_right_mode":float(right_mode_fraction(posterior_samples)),
            "seconds":perf_counter()-started,
        })
    return rows,example_samples

np.testing.assert_allclose(mmd_squared(centers,centers),0.)
rows,example_samples=run_experiment(protocol)
assert len(rows)==len(protocol.seeds) and all(np.isfinite(list(row.values())).all() for row in rows)
assert all(row["beta_mmd2"]<0.1 and row["posterior_mmd2"]<0.1 for row in rows), "samples far from the data: recheck the reverse mean and variance"
for row in rows:
    print({key:round(value,5) for key,value in row.items()})


## 5. Summarize matched effects at the seed level

Implement paired_summary(rows, treatment, baseline, seed, draws). Each row is a complete
paired training-seed result. Calculate treatment minus baseline per row. With a private NumPy
rng, use integers to resample complete differences with replacement, and compute an average
for each of draws replicates. Return (observed mean, 2.5% bootstrap quantile, 97.5% quantile).
Draws is positive and rows is nonempty. Report this as an exploratory interval for this fixture
rather than as strong evidence of generalization.


In [ ]:
def paired_summary(rows,treatment,baseline,seed,draws):
    raise NotImplementedError


In [ ]:
assert paired_summary([{"a":1,"b":3},{"a":2,"b":4}],"a","b",0,50)==(-2.,-2.,-2.)
fixture_rows=[{"a":3,"b":2},{"a":9,"b":6}]
fixture_effects=np.array([1.,3.])
fixture_indices=np.random.default_rng(7).integers(0,2,size=(4,2))
fixture_bounds=np.quantile(fixture_effects[fixture_indices].mean(axis=1),[0.025,0.975])
np.testing.assert_allclose(paired_summary(fixture_rows,"a","b",7,4),(2.,*fixture_bounds))
primary=paired_summary(rows,"conditioned_mse","blind_mse",13,1000)
secondary=paired_summary(rows,"beta_mmd2","posterior_mmd2",13,1000)
assert all(np.isfinite(primary)) and all(np.isfinite(secondary))
print("Conditioned minus blind noise MSE:",primary)
print("Variance beta minus posterior variance, biased RBF MMD²:",secondary)

print("OK")


## Inspect generated samples

The arrays in example_samples hold the reference, samples with variance beta and with the posterior variance for the
first seed. Inspect them alongside the table. If Matplotlib is installed, this optional plot
uses shared scales. The quantitative report still uses every seed.


In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("First five points per condition:", [points[:5] for points in example_samples])
else:
    figure, axes=plt.subplots(1,3,figsize=(10,3),sharex=True,sharey=True,constrained_layout=True)
    for axis, name,points in zip(axes,("Held-out reference","Variance beta","Posterior variance"),example_samples):
        axis.scatter(points[:,0],points[:,1],s=7,alpha=0.5)
        axis.set_title(name)
        axis.set_aspect("equal",adjustable="box")
    plt.show()


# Part B. Train a DDPM on MNIST on a Colab GPU (required milestone)

Upload this notebook to [Colab](https://colab.research.google.com/) and choose
Runtime > Change runtime type > T4 GPU. Run Part A first: Part B reuses your reverse_variance and
checks its torch sampler against your reverse_sample. Kaggle notebooks with a GPU are a fallback.

The run is sized for a free T4 (as of late 2026; free-tier GPUs and quotas change): roughly
20 to 30 minutes of training and about 5 minutes of sampling and evaluation, under 45 minutes in total.
These times are estimates, so record the measured times your run prints. Set SMOKE = True first for a
one-minute pipeline check (on CPU if needed), then set it back to False for the real run. The trained
weights are saved to a checkpoint file, so after a disconnect you can rerun the cells without retraining,
as long as the file is still in the Colab session. If you must shorten the run, keep train_steps
at 10,000 or more: with EMA decay 0.999 the averaged weights still carry a large share of the random
initialization after a few thousand steps (0.999 to the power 600 is about 0.55), and samples from
them are noise.

**What follows the paper and what differs.** As in Ho et al.: epsilon prediction with the simplified
loss (Algorithm 1), $T=1000$, a linear schedule from $\beta_1=10^{-4}$ to $\beta_T=0.02$, Adam with
learning rate $2\times10^{-4}$, gradient clipping at 1, an exponential moving average (EMA) of the weights
for sampling, ancestral sampling (Algorithm 2) and the fixed-variance choice of Section 3.2. Differences:
MNIST at 28×28 instead of CIFAR-10, a small two-resolution U-Net without attention, 20,000 steps at batch
size 128 instead of 800,000, EMA decay 0.999 instead of 0.9999 (the run is short), no learning-rate warmup
and mixed precision. The image metric is a Fréchet distance in the 128-dimensional feature space of a
small MNIST classifier trained here. It is an FID proxy, not FID: its values are not comparable with
published Inception-based FIDs.

In [ ]:
# Colab already includes torch, torchvision and matplotlib; this fills any gap elsewhere.
!pip install -q torch torchvision matplotlib

In [ ]:
import copy
import math
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

SMOKE = False  # True: tiny pipeline check only. Its numbers mean nothing.

@dataclass(frozen=True)
class GPUProtocol:
    timesteps: int = 1000
    beta_first: float = 1e-4
    beta_last: float = 0.02
    base_channels: int = 64
    train_steps: int = 20000
    batch_size: int = 128
    learning_rate: float = 2e-4
    ema_decay: float = 0.999
    train_seed: int = 0
    sample_seeds: tuple = (101, 202, 303)
    samples_per_seed: int = 1000
    classifier_steps: int = 1500
    checkpoint: str = "ddpm_mnist.pt"

gpu_protocol = GPUProtocol()
if SMOKE:
    gpu_protocol = GPUProtocol(timesteps=20, base_channels=16, train_steps=3, batch_size=8,
                               sample_seeds=(101, 202), samples_per_seed=16, classifier_steps=3,
                               checkpoint="ddpm_mnist_smoke.pt")
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu" and not SMOKE:
    print("No GPU found. Choose Runtime > Change runtime type > T4 GPU, or set SMOKE = True.")
print("torch", torch.__version__, "torchvision", torchvision.__version__, "device", device,
      torch.cuda.get_device_name(0) if device == "cuda" else "")
print("GPU protocol:", asdict(gpu_protocol))

def load_mnist(train):
    data = torchvision.datasets.MNIST("data", train=train, download=True)
    images = data.data.float().div(127.5).sub(1).unsqueeze(1)  # (n, 1, 28, 28) in [-1, 1]
    return images.to(device), data.targets.to(device)

train_images, train_labels = load_mnist(True)
test_images, test_labels = load_mnist(False)
gpu_betas = np.linspace(gpu_protocol.beta_first, gpu_protocol.beta_last, gpu_protocol.timesteps)
gpu_retained = np.cumprod(1 - gpu_betas)
retained_tensor = torch.tensor(gpu_retained, dtype=torch.float32, device=device)
print("train", tuple(train_images.shape), "test", tuple(test_images.shape),
      "final retained", gpu_retained[-1])

## 6. Implement the simplified training loss

Implement ddpm_loss(model, clean, t, noise, retained). clean and noise are (batch, 1, 28, 28) tensors,
t holds integer timestep indices and retained is the tensor of cumulative retentions
$\bar\alpha_t$. Form $x_t=\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\epsilon$, broadcasting
each example's retention over its channel and pixel axes, call model(x_t, t) and return the mean squared
error between the prediction and the noise. This is Algorithm 1's loss, the tensor version of your
forward_sample.

In [ ]:
def ddpm_loss(model,clean,t,noise,retained):
    raise NotImplementedError

In [ ]:
class TimeScaledEcho(nn.Module):
    def forward(self, x, t):
        return x * (t + 1).view(-1, 1, 1, 1)

generator = torch.Generator().manual_seed(0)
fixture_clean = torch.randn(3, 1, 4, 4, generator=generator)
fixture_noise = torch.randn(3, 1, 4, 4, generator=generator)
fixture_t = torch.tensor([2, 0, 1])
fixture_retained = torch.tensor([0.9, 0.5, 0.1])
a = fixture_retained[fixture_t].view(-1, 1, 1, 1)
noisy = a.sqrt() * fixture_clean + (1 - a).sqrt() * fixture_noise
expected = ((noisy * (fixture_t + 1).view(-1, 1, 1, 1) - fixture_noise) ** 2).mean()
torch.testing.assert_close(ddpm_loss(TimeScaledEcho(), fixture_clean, fixture_t, fixture_noise, fixture_retained), expected)
print("OK")

## 7. Train the U-Net (provided)

The network predicts the noise from $x_t$ and a sinusoidal timestep embedding. It has two resolutions
(28×28 and 14×14, with a 7×7 middle), residual blocks with group normalization and skip connections.
Training is skipped when a checkpoint with the same protocol already exists.

In [ ]:
def timestep_embedding(t, dim):
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
    angles = t.float()[:, None] * freqs[None]
    return torch.cat([angles.sin(), angles.cos()], dim=1)

class ResBlock(nn.Module):
    def __init__(self, cin, cout, tdim):
        super().__init__()
        self.norm1, self.conv1 = nn.GroupNorm(8, cin), nn.Conv2d(cin, cout, 3, padding=1)
        self.time = nn.Linear(tdim, cout)
        self.norm2, self.conv2 = nn.GroupNorm(8, cout), nn.Conv2d(cout, cout, 3, padding=1)
        self.skip = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()

    def forward(self, x, temb):
        h = self.conv1(F.silu(self.norm1(x))) + self.time(temb)[:, :, None, None]
        return self.conv2(F.silu(self.norm2(h))) + self.skip(x)

class SmallUNet(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.ch, tdim = ch, 4 * ch
        self.time_mlp = nn.Sequential(nn.Linear(ch, tdim), nn.SiLU(), nn.Linear(tdim, tdim))
        self.inp = nn.Conv2d(1, ch, 3, padding=1)
        self.down1 = ResBlock(ch, ch, tdim)                          # 28x28
        self.pool1 = nn.Conv2d(ch, ch, 3, stride=2, padding=1)       # to 14x14
        self.down2 = ResBlock(ch, 2 * ch, tdim)
        self.pool2 = nn.Conv2d(2 * ch, 2 * ch, 3, stride=2, padding=1)  # to 7x7
        self.mid1, self.mid2 = ResBlock(2 * ch, 2 * ch, tdim), ResBlock(2 * ch, 2 * ch, tdim)
        self.up2 = ResBlock(4 * ch, 2 * ch, tdim)
        self.up1 = ResBlock(3 * ch, ch, tdim)
        self.out = nn.Sequential(nn.GroupNorm(8, ch), nn.SiLU(), nn.Conv2d(ch, 1, 3, padding=1))

    def forward(self, x, t):
        temb = self.time_mlp(timestep_embedding(t, self.ch))
        h1 = self.down1(self.inp(x), temb)
        h2 = self.down2(self.pool1(h1), temb)
        h = self.mid2(self.mid1(self.pool2(h2), temb), temb)
        h = self.up2(torch.cat([F.interpolate(h, scale_factor=2.0, mode="nearest"), h2], 1), temb)
        h = self.up1(torch.cat([F.interpolate(h, scale_factor=2.0, mode="nearest"), h1], 1), temb)
        return self.out(h)

def train_ddpm(gp):
    torch.manual_seed(gp.train_seed)
    model = SmallUNet(gp.base_channels).to(device)
    ema = copy.deepcopy(model).eval().requires_grad_(False)
    optimizer = torch.optim.Adam(model.parameters(), lr=gp.learning_rate)
    use_amp = device == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    generator = torch.Generator(device=device).manual_seed(gp.train_seed)
    losses, running, started = [], 0.0, perf_counter()
    for step in range(1, gp.train_steps + 1):
        index = torch.randint(0, len(train_images), (gp.batch_size,), device=device, generator=generator)
        clean = train_images[index]
        t = torch.randint(0, gp.timesteps, (gp.batch_size,), device=device, generator=generator)
        noise = torch.randn(clean.shape, device=device, generator=generator)
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
            loss = ddpm_loss(model, clean, t, noise, retained_tensor)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        with torch.no_grad():
            for averaged, current in zip(ema.parameters(), model.parameters()):
                averaged.lerp_(current, 1 - gp.ema_decay)
        running += loss.detach().float()
        if step % 500 == 0 or step == gp.train_steps:
            count = step - 500 * ((step - 1) // 500)
            losses.append((step, float(running) / count))
            running = 0.0
            print(f"step {step}: mean loss {losses[-1][1]:.4f}, {perf_counter() - started:.0f} s")
    if not all(torch.isfinite(value).all() for value in ema.state_dict().values()):
        raise ValueError("training produced nonfinite weights")
    return ema, losses, perf_counter() - started

saved = torch.load(gpu_protocol.checkpoint, map_location=device) if os.path.exists(gpu_protocol.checkpoint) else None
if saved is not None and saved["protocol"] == asdict(gpu_protocol):
    ema = SmallUNet(gpu_protocol.base_channels).to(device).eval().requires_grad_(False)
    ema.load_state_dict(saved["ema"])
    losses, train_seconds = saved["losses"], saved["seconds"]
    print("Loaded checkpoint; training took", round(train_seconds), "s")
else:
    ema, losses, train_seconds = train_ddpm(gpu_protocol)
    torch.save({"ema": ema.state_dict(), "losses": losses, "seconds": train_seconds,
                "protocol": asdict(gpu_protocol)}, gpu_protocol.checkpoint)
print("parameters:", sum(value.numel() for value in ema.parameters()))

## 8. Sample with both reverse variances on the same weights

The sampler applies Algorithm 2 with the EMA weights. For each sampling seed it draws the starting
noise and every reverse-step noise from one generator, so the $\sigma_t^2=\beta_t$ and
$\sigma_t^2=\tilde\beta_t$ runs differ only in the variance. The final step adds no noise. The first
check confirms that the torch step matches your NumPy reverse_sample.

In [ ]:
def ddpm_step(state, predicted_noise, t, variance, noise):
    beta, a = float(gpu_betas[t]), float(gpu_retained[t])
    mean = (state - beta * predicted_noise / math.sqrt(1 - a)) / math.sqrt(1 - beta)
    return mean + math.sqrt(variance) * noise

fixture = np.random.default_rng(5).standard_normal((3, 2, 1, 3, 3))
for choice in ("beta", "posterior"):
    t = min(7, gpu_protocol.timesteps - 1)
    variance = reverse_variance(float(gpu_betas[t]), float(gpu_retained[t]), float(gpu_retained[t - 1]), choice)
    np.testing.assert_allclose(
        ddpm_step(*(torch.tensor(value) for value in fixture[:2]), t, variance, torch.tensor(fixture[2])).numpy(),
        reverse_sample(fixture[0], fixture[1], float(gpu_betas[t]), float(gpu_retained[t]), variance, fixture[2]),
        rtol=1e-10)

@torch.no_grad()
def sample(model, gp, seed, choice, count):
    generator = torch.Generator(device=device).manual_seed(seed)
    state = torch.randn((count, 1, 28, 28), device=device, generator=generator)
    for t in range(gp.timesteps - 1, -1, -1):
        steps = torch.full((count,), t, device=device, dtype=torch.long)
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=device == "cuda"):
            prediction = model(state, steps).float()
        previous = float(gpu_retained[t - 1]) if t else 1.0
        variance = reverse_variance(float(gpu_betas[t]), float(gpu_retained[t]), previous, choice)
        noise = torch.randn(state.shape, device=device, generator=generator) if t else torch.zeros_like(state)
        state = ddpm_step(state, prediction, t, variance, noise)
    return state.clamp(-1, 1)
print("OK")

## 9. Train the feature classifier and define the FID proxy

The classifier is trained on the MNIST training images only. The reference images for the Fréchet
distance are held-out test images, which neither network saw. The distance between two disjoint
real test subsets of the same size is the noise floor. Report it next to the generated results. Class
entropy (in bits, at most $\log_2 10\approx3.32$) of the classifier's predicted labels on samples
checks whether samples cover the digits.

In [ ]:
class FeatureClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Linear(64 * 7 * 7, 128), nn.ReLU())
        self.head = nn.Linear(128, 10)

    def forward(self, x):
        return self.head(self.features(x))

def train_classifier(gp):
    torch.manual_seed(gp.train_seed + 1)
    net = FeatureClassifier().to(device)
    optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)
    generator = torch.Generator(device=device).manual_seed(gp.train_seed + 1)
    for _ in range(gp.classifier_steps):
        index = torch.randint(0, len(train_images), (256,), device=device, generator=generator)
        loss = F.cross_entropy(net(train_images[index]), train_labels[index])
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
    return net.eval().requires_grad_(False)

@torch.no_grad()
def classify(images):
    features = torch.cat([classifier.features(batch) for batch in images.split(500)])
    return features.double().cpu().numpy(), classifier.head(features).argmax(1).cpu().numpy()

def frechet_distance(a, b):
    difference = a.mean(axis=0) - b.mean(axis=0)
    cov_a, cov_b = np.cov(a, rowvar=False), np.cov(b, rowvar=False)
    # trace of the matrix square root of cov_a @ cov_b: its eigenvalues are real and nonnegative
    # in exact arithmetic, so clip rounding error before the square root
    eigenvalues = np.linalg.eigvals(cov_a @ cov_b).real.clip(min=0)
    return float(difference @ difference + np.trace(cov_a) + np.trace(cov_b) - 2 * np.sqrt(eigenvalues).sum())

def class_entropy(labels):
    shares = np.bincount(labels, minlength=10) / len(labels)
    shares = shares[shares > 0]
    return float((shares * np.log2(1 / shares)).sum())

check = np.random.default_rng(0).standard_normal((400, 3))
assert abs(frechet_distance(check, check)) < 1e-6
np.testing.assert_allclose(frechet_distance(check, check + 2.0), 12.0, rtol=1e-6)

classifier = train_classifier(gpu_protocol)
_, test_predictions = classify(test_images)
print("classifier test accuracy:", float(np.mean(test_predictions == test_labels.cpu().numpy())))

## 10. Run the variance ablation and report it by seed

Each sampling seed gives one matched pair of runs. The paired summary reuses your paired_summary from
Part A. Three sampling seeds measure sampling noise only: the network was trained once, so
training-seed variation is not measured. Say so in the paper.

In [ ]:
count = gpu_protocol.samples_per_seed
reference_features, _ = classify(test_images[:count])
floor_features, _ = classify(test_images[count:2 * count])
noise_floor = frechet_distance(floor_features, reference_features)
gpu_rows, grid_images = [], {}
for seed in gpu_protocol.sample_seeds:
    row = {"seed": seed}
    for choice in ("beta", "posterior"):
        started = perf_counter()
        images = sample(ema, gpu_protocol, seed, choice, count)
        features, labels = classify(images)
        row[f"{choice}_fd"] = frechet_distance(features, reference_features)
        row[f"{choice}_entropy"] = class_entropy(labels)
        row[f"{choice}_seconds"] = perf_counter() - started
        grid_images.setdefault(choice, images[:64].float().cpu())
    gpu_rows.append(row)
    print({key: round(value, 3) for key, value in row.items()})
assert all(np.isfinite(list(row.values())).all() for row in gpu_rows)
print("real-versus-real noise floor:", round(noise_floor, 3), "training seconds:", round(train_seconds))
print("Variance beta minus posterior variance, FID proxy:",
      paired_summary(gpu_rows, "beta_fd", "posterior_fd", 13, 1000))

In [ ]:
import matplotlib.pyplot as plt

figure, axes = plt.subplots(1, 2, figsize=(10, 5.4), constrained_layout=True)
titles = {"beta": "Variance beta", "posterior": "Posterior variance"}
for axis, (choice, images) in zip(axes, grid_images.items()):
    grid = torchvision.utils.make_grid(images, nrow=8, normalize=True, value_range=(-1, 1))
    axis.imshow(grid.permute(1, 2, 0).numpy())
    axis.set_title(f"{titles[choice]}, seed {gpu_protocol.sample_seeds[0]}")
    axis.axis("off")
figure.savefig("ddpm_mnist_samples.png", dpi=150)
plt.show()

## Submit the paper in this notebook

Write a short paper in the usual format. Put every number next to its seeds and units.

1. **Abstract.** The claim you test (Ho et al., Section 3.2: $\sigma_t^2=\beta_t$ and $\tilde\beta_t$
   give similar results), your scope (2-D mixture and MNIST, not CIFAR-10) and the main result.
2. **Method.** The forward marginal, the simplified loss (Algorithm 1), the reverse step and both
   variances (Algorithm 2), with citations, and the differences from the paper listed in Part B.
3. **Experimental setup.** Both protocols as printed, environment and versions, hardware, training
   and sampling seeds, evaluation data, the declared smallest meaningful differences and measured runtimes.
4. **Results table.** Part A: per-seed noise MSE, MMD² and mode fractions for both ablations, with the
   paired effects and bootstrap intervals. Part B: per-seed FID proxy and class entropy for both
   variances, the real-versus-real noise floor, the paired effect, the classifier's test accuracy and
   the sample grid.
5. **Ablations.** Interpret the time-conditioning ablation and the variance ablation separately. Say
   whether your result agrees with the paper's statement and how much the evidence can support.
6. **Limitations and ethics.** One trained MNIST model (training-seed variation not measured), few
   seeds, an FID proxy that is not comparable with published FID, the fixed MMD bandwidth and the small
   synthetic population. MNIST is a public dataset of handwritten digits; state its source and licence
   as you find them, and avoid claims about other image benchmarks or deployment.
7. **Reproduction appendix.** Protocol output, versions, checkpoint name, anything that failed and
   one follow-up comparison that would settle an open question.

Write your paper here.